# 第8回A 演習: 言語モデルの実装 (ブラウザ版)

スライド: [第8回A ブラウザ版](../08a-llm-webllm/) / Colab 版は[こちら](https://colab.research.google.com/gist/3tty0n/1c0f3cf4d3f4fb2f30354677156e75ea/08a_llm.ipynb)

Qwen3.5-0.8B をブラウザの中 (WebGPU) で動かします。コードはすべて Python です。

- **テンソル演算** (softmax / topk / 温度 / top-p) → `torch` (torch-pyodide)
- **モデルの生成** (LLM 本体) → WebLLM (Pyodide が内部でブリッジ)
- 課題3 と 発展課題は最後の `score()` で自己確認できる

- **WebGPU 対応ブラウザ** (Chrome / Edge の最新版) で開くこと
- 初回だけモデルと torch ランタイムをダウンロード (約 0.6 GB、Wi-Fi 推奨)。2回目からキャッシュ
- 生成中はタブを開いたままにする
- 提出前に **出力を残したまま** `File → Download` で .ipynb を保存

**torch について**: torch-pyodide は本家 PyTorch のブラウザ移植 (実験版) です。
transformers や学習済みモデルの重みは含まないため、LLM 本体は WebLLM を使います。
Colab 版との違いは、`torch.topk`/`torch.sort` がタプルを返すことと、
テンソルを添え字にできない (`.item()` でスカラー化) ことです。

---

## セットアップ

下の2つのセルを実行する。初回はダウンロードに数分かかる。

In [ ]:
# [SKIP] 採点の対象外
# ---- セットアップ (最初に1回だけ実行) ----
# WebGPU 対応ブラウザ (Chrome / Edge の最新版) で実行すること
import js
from js import Object
from pyodide.ffi import to_js, run_sync

# torch 本体 (torch-pyodide) を読み込む
_manifest = await js.Function(
    'u', 'return (async()=>{const r=await fetch(u); return await r.json()})(u)')(
    'https://celsowm.github.io/torch-pyodide/latest.json')
_runtime = await js.Function('u', 'return import(u)')(_manifest.runtimeUrl)
_runtime.installTorchRuntime(js.globalThis)

import micropip
await micropip.install(str(_manifest.wheelUrl))
import torch
print("torch:", torch.__version__, "| GPU:", torch.cuda.is_available())

# LLM (WebLLM) を読み込む
webllm = await js.Function('u', 'return import(u)')(
    'https://esm.run/@mlc-ai/web-llm@0.2.84',
)
MODEL = "Qwen3.5-0.8B-q4f16_1-MLC"
print("キャッシュ済み:", bool(await webllm.hasModelInCache(MODEL)))

def _progress(r):
    print(f"[読み込み中 {r.progress:>4.0%}] {r.text}")

engine = await webllm.CreateMLCEngine(
    MODEL,
    to_js({"initProgressCallback": _progress}, dict_converter=Object.fromEntries),
)
print("準備完了:", MODEL)

# ---- ヘルパー ----
def sample_next(logits, temperature=0.8, k=50):
    # 温度で割って softmax → 上位 k 個に絞ってから重み付きで選ぶ
    probs = torch.softmax(logits / temperature, dim=-1)
    values, indices = torch.topk(probs, k)
    return indices[torch.multinomial(values, 1).item()].item()

def sample_next_top_p(logits, temperature=0.8, p=0.9):
    # 確率の高い順に累積し、合計が p に達した候補だけ残す
    probs = torch.softmax(logits / temperature, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cumsum = torch.cumsum(sorted_probs)
    mask = cumsum - sorted_probs < p
    return sorted_idx[mask][torch.multinomial(sorted_probs[mask], 1).item()].item()

def to_payload(messages, **opts):
    # Python の辞書を JavaScript のオブジェクトに変換する
    return to_js({"messages": messages, **opts},
                 dict_converter=Object.fromEntries)

def reply_text(reply):
    # 応答オブジェクトからテキストを取り出す
    return reply.choices[0].message.content.strip()

def prompt_tokens(reply):
    return reply.usage.prompt_tokens

def completion_tokens(reply):
    return reply.usage.completion_tokens

def total_tokens(reply):
    return reply.usage.total_tokens

async def _chat_reply(messages, **opts):
    # 応答オブジェクトをそのまま返す (内部用。run_sync で同期化する)
    opts.setdefault("max_tokens", 32)
    opts.setdefault("enable_thinking", False)  # 思考モード off (簡潔な応答)
    return await engine.chat.completions.create(to_payload(messages, **opts))

def chat_reply(messages, **opts):
    # 応答オブジェクトを返す (同期)。reply_text などで取り出す
    return run_sync(_chat_reply(messages, **opts))

def chat(messages, **opts):
    # 会話履歴 messages を渡して、応答の文字列を返す (同期)
    return reply_text(chat_reply(messages, **opts))

def ask(prompt, **opts):
    # 1往復の応答の文字列を返す (同期。chat の薄いラッパー)
    return chat([{"role": "user", "content": prompt}], **opts)

async def stream(messages, **opts):
    # 応答をトークン単位で yield する (発展4 用)
    opts.setdefault("max_tokens", 64)
    opts.setdefault("enable_thinking", False)
    gen = await engine.chat.completions.create(
        to_payload(messages, stream=True, **opts))
    async for chunk in gen:
        piece = chunk.choices[0].delta.content or ""
        if piece:
            yield piece

In [ ]:
#@title 自動セットアップ (実行するだけ。中身は隠しています)
# [SKIP] 採点の対象外
# ---- 自己確認の準備 (check / score)。1回実行すればよい ----
import pathlib
import urllib.request

if not pathlib.Path('autograde.py').exists():
    try:
        urllib.request.urlretrieve(
            'https://3tty0n.github.io/2026-scripting-lab/grading/autograde.py', 'autograde.py')
    except Exception as e:
        print('取得できませんでした:', e)

from autograde import check, score
TASK = '08a_web'


---

## API 早見表

詰まったらここに戻る。課題で使うのはこの表の範囲だけ。

| 使うもの | 意味 |
|---|---|
| `ask(prompt, **opts)` | 1往復。応答の文字列を返す (同期) |
| `chat(messages, **opts)` | 履歴 `messages` を渡して応答を返す (同期) |
| `chat_reply(messages, **opts)` | 応答オブジェクトを返す (同期) |
| `reply_text(reply)` | 応答オブジェクトからテキストを取り出す |
| `prompt_tokens(reply)` / `completion_tokens(reply)` / `total_tokens(reply)` | 応答のトークン数 |
| `stream(messages, **opts)` | 応答をトークン単位で yield (**async**。`async for` で回す・発展3) |
| `engine` | 読み込んだモデル本体 (Qwen3.5-0.8B) |

> **注意**: `chat` / `ask` / `chat_reply` は**同期** (await 不要)。
> だけど `stream` だけは **async** なので `async for` で回す。

**messages**: `{"role": "system" | "user" | "assistant", "content": "..."}` のリスト。

**opts** に渡せる生成オプション:

| オプション | 意味 |
|---|---|
| `max_tokens` | 作るトークン数の上限 |
| `temperature` | ランダム性。0 で決め打ち、高いほどばらける |
| `top_p` | 累積確率で候補を絞る |
| `presence_penalty` / `frequency_penalty` | 繰り返しを減らす |
| `enable_thinking` | 思考モード (デフォルト off。on にすると考えてから答える) |
| `stream` | 応答をトークン単位で受け取る (発展4) |

In [ ]:
# [SKIP] 採点の対象外
# ---- 動作確認 ----
print("応答:", ask("日本の首都は？"))

# 温度の効果 (Colab 版と同じ結果になる)
logits = torch.tensor([3.0, 1.0])
print(torch.softmax(logits / 0.1, dim=-1))  # 低温は決め打ちに近い
print(torch.softmax(logits / 2.0, dim=-1))  # 高温はばらける

---

## 課題1: トークンを数える

`prompt_tokens(reply)` と `completion_tokens(reply)` で、
入力と出力それぞれのトークン数を表示しよう。

使うもの: `reply_text(reply)` / `prompt_tokens(reply)` / `completion_tokens(reply)`

```
出力例:
続き: おじいさんとおばあさんが...
入力のトークン数: 7
作ったトークン数: 16
```

In [ ]:
# 課題1: 入力と出力それぞれのトークン数を表示しよう
reply = chat_reply(
    [{"role": "user", "content": "むかしむかし、あるところに"}],
    max_tokens=16)

# TODO: reply_text(reply) で続きを、prompt_tokens / completion_tokens で数を表示
print("続き:", ...)
print("入力のトークン数:", ...)
print("作ったトークン数:", ...)

---

## 課題2: greedy の確認

`temperature: 0` でランダム性が消え、毎回同じ出力になる。
同じ入力を2回投げて、結果が一致することを確かめよう。

使うもの: `temperature: 0` のオプション

In [ ]:
# 課題2: temperature 0 で、同じ入力から同じ出力が返ることを確かめよう
def ask_greedy(prompt):
    return chat([{"role": "user", "content": prompt}],
                      temperature=0, max_tokens=24)

a = ask_greedy("好きな食べ物は")
b = ask_greedy("好きな食べ物は")

# TODO: a と b を表示し、a == b が True になることを確認

# TODO: ここに a と b と a == b を表示する


---

## 課題3: 温度と top-k、top-p

確率分布の形は温度で変わる。ダミーの logits に `sample_next` を
適用し、`0.3 / 0.8 / 1.5` の3つで選ばれる候補を比べよう。
`sample_next_top_p` でも p の違いを試そう。

使うもの: `sample_next(logits, temperature=t, k=k)` / `sample_next_top_p(logits, temperature=0.8, p=p)` (torch)

```
出力例 (同じ乱数でも、温度や p によって選ばれ方が変わる):
T=0.3: ... (強い候補ばかり)
T=1.5: ... (低い候補も混ざる)
p=0.9: ... (候補が多い)
p=0.5: ... (候補が少ない)
```

In [ ]:
# 課題3: 温度と top-k、top-p
logits = torch.randn(100)  # ダミーの logits (語彙サイズ 100)

# 温度と top-k: 0.3 / 0.8 / 1.5 で選ばれる候補を比べよう
for t in [0.3, 0.8, 1.5]:
    picked = [int(sample_next(logits, temperature=t, k=10)) for _ in range(8)]
    print(f"T={t}: {picked}")

# top-p: 累積確率 p の違いで候補の広がりを比べよう
print("p=0.9:", [int(sample_next_top_p(logits, temperature=0.8, p=0.9)) for _ in range(8)])
print("p=0.5:", [int(sample_next_top_p(logits, temperature=0.8, p=0.5)) for _ in range(8)])

**まとめ** (書き換えてよい): 低温は強い候補ばかり、高温は低い候補も混ざる。

---

## 課題4: 簡易チャットアプリを作る

モデルは履歴を覚えない。過去のやりとりを毎回 `messages` に全部入れて
渡すのが呼び出し側の仕事。履歴を保持する「チャットボット」`make_chat()` を
作り、好きな話題で3往復以上会話しよう。`input()` は環境によって使えないので、
あらかじめ用意した質問をループで回す形にする。

使うもの: `chat(messages, **opts)` と クロージャ (`history` を閉じ込める)

- 履歴は `make_chat()` の中だけで持ち、毎回全部渡す
- 発展: `make_chat()` にキャラクター設定 (system) を渡せるようにする


In [ ]:
# 課題4: 簡易チャットアプリを作ろう
def make_chat(system_prompt="あなたは簡潔に答えるアシスタントです。"):
    """会話の履歴を保持するチャットボットを返す"""
    history = [{"role": "system", "content": system_prompt}]

    def say(text):
        history.append({"role": "user", "content": text})
        # TODO: chat(history, max_tokens=64) で応答の文字列を得る (同期なので await 不要)
        reply = ...
        # TODO: history に assistant の応答を追加し、応答を返す
        ...
        return reply

    return say

bot = make_chat()
for q in ["日本の首都は？", "そこの名所を教えて", "おすすめをひとつ教えて"]:
    a = bot(q)
    print(f"あなた: {q}")
    print(f"アシスタント: {a}\n")

---

## 発展: チャットアプリを本物に近づける

課題4 の `make_chat` を、発展1〜4 で段階的に改良していく。
それぞれまず「部品となる純関数」を書き、それを `make_chat` に組み込む。
部品は `score()` で採点される。

- 発展1: 繰り返しの検出 → 単調さ対策
- 発展2: コンテキストと圧縮 → 長い会話への対応
- 発展3: ストリーミング表示 → 生成を体感
- 発展4: エフォート → 生成の丁寧さを選ぶ

---

### 発展1: 繰り返しの検出

応答が同じフレーズを繰り返すと単調になる。テキストの中に、
n 語のまとまりが2回以上現れたら `True` を返す `has_repeat(text, n)` を書こう。

使うもの: `text.split()` と 集合 (`set`) または `in` 判定

ヒント: 先頭から n 語ずつのチャンクを作り、「見たことがあるか」を覚えておく

**チャンクと set の早見表**:

```python
words = text.split()          # 文字列 -> 語のリスト
chunk = tuple(words[i:i + n]) # n 語のまとまり (スライス)。タプルにすると set に入れられる
seen = set()                  # 「見たことがあるもの」の集合
chunk in seen                 # 含まれているか (True / False)
seen.add(chunk)               # 追加
```

In [ ]:
# 発展1: 繰り返しの検出
def has_repeat(text, n=3):
    # TODO: n 語のまとまりが2回以上出たら True を返す
    ...

print(has_repeat("I like cats. I like cats. Yes", 3))  # True (I like cats が繰り返し)
print(has_repeat("the quick brown fox", 3))             # False

---

### 発展2: コンテキストと圧縮

会話が長くなると入力上限に近づく。履歴のトークン数を概算する
`estimate_tokens(messages)` と、上限を超えたら古いやりとりを捨てる
`compact(messages, limit)` を書こう。`system` は残すこと。

使うもの: `len(...)` による文字数概算、スライス `messages[1:3]` で1往復を捨てる

ヒント: 1トークン ≈ 日本語2文字とみなして概算してよい

In [ ]:
# 発展2: コンテキストと圧縮
def estimate_tokens(messages):
    # TODO: 全 content の文字数からトークン数を概算する (例: 文字数 // 2)
    ...

def compact(messages, limit):
    # TODO: 概算が limit を超える間、古い1往復 (2セル) を捨てる。system は残す
    ...

msgs = [
    {"role": "system", "content": "あなたは簡潔に答えるアシスタントです。"},
    {"role": "user", "content": "あ" * 200},
    {"role": "assistant", "content": "い" * 200},
    {"role": "user", "content": "う"},
]
print(len(compact(msgs, 50)))   # system だけ残るはず

---

### 発展3: ストリーミング表示

生成されたトークンがポツポツ返ってくるのを体感しよう。断片のリストを
1つの文字列にまとめる `collect(pieces)` を書き、`stream()` と組み合わせて
出たそばから表示する `show(messages)` を作る。

使うもの: 文字列の連結 (`"".join(pieces)` か `+=`)

In [ ]:
# 発展3: ストリーミング表示
def collect(pieces):
    # TODO: 断片のリストを1つの文字列にまとめる
    ...

async def show(messages, **opts):
    pieces = []
    async for piece in stream(messages, **opts):  # 1トークンずつ
        pieces.append(piece)
        print(piece, end="", flush=True)
    return collect(pieces)

print()
print(await show([{"role": "user", "content": "5行の自己紹介の詩を書いて"}]))

---

### 発展4: エフォート (low / medium / high)

生成の「丁寧さ」を effort で切り替える。WebLLM に reasoning effort 相当の
パラメータは無いので、自分で設定を組み立てる。effort ごとの設定を返す
`effort_opts(effort)` を書き、`make_chat` に組み込もう。

使うもの: dict の参照 `EFFORT[effort]`

ヒント: low は短く・ばらける (温度高め)、high は長く・堅く (温度低め)

In [ ]:
# 発展4: エフォート (low / medium / high)
EFFORT = {
    "low":    {"max_tokens": 32,  "temperature": 1.0},
    "medium": {"max_tokens": 64,  "temperature": 0.8},
    "high":   {"max_tokens": 128, "temperature": 0.3},
}

def effort_opts(effort):
    # TODO: EFFORT[effort] を返す
    ...

def make_chat(system_prompt="あなたは簡潔に答えるアシスタントです。", effort="medium"):
    opts = effort_opts(effort)
    history = [{"role": "system", "content": system_prompt}]

    def say(text):
        history.append({"role": "user", "content": text})
        reply = chat(history, **opts)
        history.append({"role": "assistant", "content": reply})
        return reply

    return say

bot = make_chat(effort="high")
for q in ["日本の首都は？", "そこの人口は？"]:
    print(bot(q))

---

## 自己確認

`score()` で課題3 と 発展課題 (発展1〜4) をまとめて確かめる。何度やってもよい。
`NG` の行に、次に何をすればいいかが出る。

In [ ]:
# [SKIP] 採点の対象外
score()

---

## 提出

提出は kibaco から。`score()` は下書き確認用。

1. 上から実行し、出力を残す (発展課題まで全部)
2. `File → Download` で .ipynb を保存
3. 締切までに kibaco へ出す